In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# DataLoader

In [64]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv
# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")
DATA_DIR = os.getenv("DATA_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import your newly structured module
from kg_commit.knowledge.dataloader import CommitDataLoader, JITDatasetAdapter
from kg_commit.knowledge.parsers import FilteredCommitParser, IdentityCommitParser
from kg_commit.knowledge.utils import CommitPayloadPrinter

In [65]:
# 1. Setup paths
CSV_PATH = f"{DATA_DIR}/apachejit/projects/apache_groovy.csv"

In [66]:
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    # Iterate through all direct items in the repos folder
    for item in base_path.iterdir():
        if item.is_dir():
            # Check if it contains a hidden .git directory to verify it's a real repo
            git_dir = item / ".git"
            if git_dir.exists():
                # Reconstruct the project key name (e.g., "apache/groovy")
                project_key = f"{prefix}{item.name.lower()}"
                
                # Assign the absolute string path as the value
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

# Print out your freshly discovered mappings
print("📂 Automatically generated REPO_MAP mappings:")
print("-" * 50)
for project, local_path in REPO_MAP.items():
    print(f"  '{project}'")
print("-" * 50)

📂 Automatically generated REPO_MAP mappings:
--------------------------------------------------
  'apache/activemq'
  'apache/camel'
  'apache/cassandra'
  'apache/flink'
  'apache/groovy'
  'apache/hadoop'
  'apache/hadoop-hdfs'
  'apache/hadoop-mapreduce'
  'apache/hbase'
  'apache/hive'
  'apache/ignite'
  'apache/kafka'
  'apache/spark'
  'apache/zeppelin'
  'apache/zookeeper'
--------------------------------------------------


In [68]:
# 2. Instantiate systems
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = IdentityCommitParser()

# 3. Pull a random commit record
all_records = list(adapter.stream_records(CSV_PATH))
random_record = random.choice(all_records)

# 4. Extract rich payload parameters from repository metadata
git_payload = loader.fetch_commit_data(
    project=random_record["project"], 
    commit_id=random_record["commit_id"]
)

if git_payload:
    full_payload = {**random_record, **git_payload}
    
    # 5. Print out the raw dictionary dynamically via our class utility
    CommitPayloadPrinter.print_payload(full_payload)
    
    # 6. Parse and check entity configurations
    parsed_results = parser.parse(full_payload)

✅ Success! Raw payload retrieved with customized features.
KEY                       | VALUE
commit_id                 | d28483e97deca733913c618a9e4230fc6ed8495d
project                   | apache/groovy
buggy                     | False
fix                       | False
year                      | 2018
author_date               | 1538815322
message                   | Trivial refactoring: extract method `addPhaseOperations` ...
diff                      | [Length: 1699 chars] -> @@ -176,6 +176,25 @@ public class CompilationUnit extends ProcessingUnit {          this.optimizer =...
parents                   | ['30c11c80c112c78760b23cf2b0f3e54cd5df1a0a']
parents_length            | 1
linked_issues             | []
containing_branches       | ['master']
author_name               | Daniel Sun
author_email              | sunlan@apache.org
authored_timestamp        | 1538815322
authored_datetime         | 2018-10-06T16:42:02+08:00
committer_name            | Daniel Sun
committed_datetime   

In [69]:
# 1. Initialize our components
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# 2. Test reading records via the Adapter
print("--- Testing CSV Adapter Filtering ---")
records_stream = adapter.stream_records(CSV_PATH)

# Take the first two rows for validation
for i, record in enumerate(records_stream):
    if i >= 2: 
        break
    print(f"\nRecord #{i+1} parsed from CSV:")
    print(record)
    
    # 3. Use the filtered metadata to fetch Git info right away
    print(f"Fetching Git text data for commit: {record['commit_id']}...")
    git_payload = loader.fetch_commit_data(project=record['project'], commit_id=record['commit_id'])
    
    if git_payload:
        print(f"✅ Extracted Message length: {len(git_payload['message'])} chars")
        print(f"✅ Extracted Diff length: {len(git_payload['diff'])} chars")

--- Testing CSV Adapter Filtering ---

Record #1 parsed from CSV:
{'commit_id': '7b8480744ea6e6fb41efd4329bb470c8f3c763db', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1070355653'}
Fetching Git text data for commit: 7b8480744ea6e6fb41efd4329bb470c8f3c763db...
✅ Extracted Message length: 190 chars
✅ Extracted Diff length: 11929 chars

Record #2 parsed from CSV:
{'commit_id': '192b631e7be302ecde822546ba70a9853ddbda01', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1063298262'}
Fetching Git text data for commit: 192b631e7be302ecde822546ba70a9853ddbda01...
✅ Extracted Message length: 135 chars
✅ Extracted Diff length: 613 chars


# Gitpython Commit Object

In [75]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# Grab a random commit to inspect
record = random.choice(list(adapter.stream_records(CSV_PATH)))
repo = loader._get_repo(record["project"])
commit_obj = repo.commit(record["commit_id"])

print(f"🔬 Dissecting GitPython Commit Object for ID: {commit_obj.hexsha}\n")
print("=" * 60)

properties_list = []
methods_list = []

# Analyze every attribute on the live object
for name, value in inspect.getmembers(commit_obj):
    if name.startswith('_'): 
        continue  # Skip private attributes
        
    try:
        if inspect.ismethod(value) or inspect.isroutine(value):
            methods_list.append(name)
        else:
            properties_list.append((name, type(value).__name__))
    except Exception:
        properties_list.append((name, "Unknown/Unloaded Property"))

print("📋 DATA FIELDS & PROPERTIES AVAILABLE:")
print("-" * 40)
for prop, data_type in sorted(properties_list):
    print(f"  {prop:<25} [Type: {data_type}]")

print("\n⚙️ EXECUTABLE METHODS AVAILABLE:")
print("-" * 40)
for method in sorted(methods_list):
    print(f"  {method}()")

🔬 Dissecting GitPython Commit Object for ID: f4f8e18e72ef3f3c9c4391887a2610d7eba84e76

📋 DATA FIELDS & PROPERTIES AVAILABLE:
----------------------------------------
  INDEX                     [Type: DiffConstants]
  Index                     [Type: DiffConstants]
  NULL_BIN_SHA              [Type: bytes]
  NULL_HEX_SHA              [Type: str]
  NULL_TREE                 [Type: DiffConstants]
  TIobj_tuple               [Type: _GenericAlias]
  TYPES                     [Type: tuple]
  author                    [Type: Actor]
  author_tz_offset          [Type: int]
  authored_date             [Type: int]
  authored_datetime         [Type: datetime]
  binsha                    [Type: bytes]
  co_authors                [Type: list]
  committed_date            [Type: int]
  committed_datetime        [Type: datetime]
  committer                 [Type: Actor]
  committer_tz_offset       [Type: int]
  conf_encoding             [Type: str]
  data_stream               [Type: OStream]
  default

# Parsers

In [73]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = IdentityCommitParser()

print(f"Reading records from {CSV_PATH}...")
all_records = list(adapter.stream_records(CSV_PATH))

if not all_records:
    print("❌ No records found in the metadata file.")
else:
    random_record = random.choice(all_records)
    print(f"🎲 Randomly selected commit ID: {random_record['commit_id']} from {random_record['project']}")
    
    print("Extracting payload from local git repository...")
    git_payload = loader.fetch_commit_data(
        project=random_record["project"], 
        commit_id=random_record["commit_id"]
    )
    
    if git_payload:
        full_payload = {**random_record, **git_payload}
        
        print("\n=================== RAW COMMIT DIFF ===================")
        print(full_payload.get("diff", "No diff available for this commit."))
        print("========================================================\n")
        
        print("Executing simultaneous parse cycle...")
        results = parser.parse(full_payload)
        
        print("\n=== Extracted Knowledge Graph Entities (Random Sample) ===")
        print(json.dumps(results, indent=10))
    else:
        print("❌ Could not extract data from the Git repository. Verify your local paths match.")

Reading records from E:/Projects/kgcommit/data/apachejit/projects/apache_groovy.csv...
🎲 Randomly selected commit ID: 113ced10875e6a6e99bd4c4783c8cd17845a2730 from apache/groovy
Extracting payload from local git repository...

=================== RAW COMMIT DIFF ===================
@@ -1,134 +0,0 @@
-/*
- * Copyright 2008 the original author or authors.
- *
- * Licensed under the Apache License, Version 2.0 (the "License");
- * you may not use this file except in compliance with the License.
- * You may obtain a copy of the License at
- *
- *     http://www.apache.org/licenses/LICENSE-2.0
- *
- * Unless required by applicable law or agreed to in writing, software
- * distributed under the License is distributed on an "AS IS" BASIS,
- * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
- * See the License for the specific language governing permissions and
- * limitations under the License.
- */
-
-package org.codehaus.groovy.ast;
-
-import org.codehaus.groovy.con

# Gitpython Exploration

In [74]:
import git

# 1. Access the repository
repo = git.Repo(REPO_MAP['apache/hive']) 
commit = repo.head.commit

# 2. Compare the HEAD commit to its parent
if commit.parents:
    diffs = commit.parents[0].diff(commit, create_patch=True)
    
    if len(diffs) > 0:
        d = diffs[0]
        print(f"--- Dynamically Inspecting All Diff Attributes ---")
        
        # 'dir(d)' returns all attributes/methods
        # We filter out private methods (starting with _) to keep it clean
        for attr in dir(d):
            if not attr.startswith('_'):
                try:
                    value = getattr(d, attr)
                    # Only print if it's not a bound method (keeps the output readable)
                    if not callable(value):
                        print(f"{attr:20}: {value}")
                except Exception:
                    print(f"{attr:20}: <Could not access>")
    else:
        print("No changes in this commit.")
else:
    print("Root commit; no parent to diff against.")

--- Dynamically Inspecting All Diff Attributes ---
NULL_BIN_SHA        : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
NULL_HEX_SHA        : 0000000000000000000000000000000000000000
a_blob              : 59cd3c9a24920503afa714e19541f915226a62c4
a_mode              : 33188
a_path              : ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java
a_rawpath           : b'ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java'
b_blob              : 6d03bb5ea5d177b8cb4b40987ca82c9a617ce987
b_mode              : 33188
b_path              : ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java
b_rawpath           : b'ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java'
change_type         : None
copied_file         : False
deleted_file        : False
diff                : b'@@ -578,6 +578,10 @@ private static TypeInfo columnVectorTypeToTypeInfo(Type type) {\n   

C:\Users\Behnam\AppData\Local\Temp\ipykernel_11940\1356399266.py:20: DeprecationWarning: Diff.renamed is deprecated, use Diff.renamed_file instead
  value = getattr(d, attr)


In [63]:
import git

def find_first_rename(repo_path):
    repo = git.Repo(repo_path)
    
    # Iterate through commits starting from the most recent
    for commit in repo.iter_commits():
        if not commit.parents:
            continue
            
        # Get diffs compared to the first parent
        diffs = commit.parents[0].diff(commit)
        
        for d in diffs:
            # Check for Rename type
            if d.a_path != d.b_path:
                print(f"--- Rename Found in Commit {commit.hexsha[:7]} ---")
                print(f"Old Path : {d.a_path}")
                print(f"New Path : {d.b_path}")
                print(f"Similarity Score: {d.score}/100")
                return d # Return the first one found
                
    print("No renames found in recent history.")
    return None

# Usage
rename_diff = find_first_rename(REPO_MAP['apache/hive'])

--- Rename Found in Commit 13f3208 ---
Old Path : .github/workflows/docker-GA-images.yml
New Path : .github/workflows/docker-images.yml
Similarity Score: 92/100
